# Kestrel — Pretraining Dataset Audit & Selection Masterclass

> **Author Perspective:** Written as a practical, reproducible **LLM Data Engineering** guide for [**Kestrel**](https://github.com/MohammedKamalAlsyd/kestrel) — a modern **350M–700M parameter LLM** engineered to outperform OpenAI's **1.5B GPT-2 XL** across all core GPT-2 benchmarks (**WikiText-103**, **LAMBADA**, **HellaSwag**, and **ARC-Easy**) using **2–4× fewer parameters** and **zero benchmark contamination**.

---

### Why Does Pretraining Data Selection Matter So Much for Small LLMs?

At **350M–700M parameters**, a language model is **capacity-constrained**. If 35% of your training corpus is cookie banners, scraped footers, repeated SEO templates, or garbled PDF text, your weights are wasted memorizing noise instead of learning English fluency, world knowledge, and scientific reasoning.

In 2025–2026 LLM research (*FineWeb-Edu*, *DCLM*, *Nemotron-CLIMB*, *SmolLM2*, *FinePDFs*), **data quality and mixture ratios matter more than almost any architecture tweak**.

### Plain-English Glossary (Only High-Signal Metrics)

| Term | What It Means in Plain English | Why We Measure It Here |
| :--- | :--- | :--- |
| **Multi-Shard Random Sampling** | Randomly picking `8` remote shards across the entire dataset + shuffling inside each shard (`buffer_size=2,000`). | Eliminates opening-shard prefix bias (e.g., date/domain ordering) without downloading terabytes on Kaggle. |
| **Token (`tiktoken` `gpt2`)** | A sub-word chunk (~0.75 English words) that the LLM actually reads. | Ensures fair apples-to-apples slices (`3.5M tokens` per dataset) even when PDFs are 10× longer than web pages. |
| **LZ4 Compression Ratio** | `compressed_bytes / raw_bytes` using ultra-fast `lz4.block`. | Clean English prose sits in the **0.65–0.80 "Goldilocks Zone"**. `<0.65` = repetitive spam; `>0.80` = garbled OCR/noise. |
| **Gopher Repetition Rate** | % of a document's characters covered by repeated 2–10-word phrases (DeepMind Gopher / FineWeb standard). | Catches within-document copy-paste loops without unfairly penalizing long PDFs. |
| **Cross-Doc 7-Gram Boilerplate** | 7-word phrases appearing across `>= 0.5%` (min 10) of distinct documents. | Catches site-wide cookie banners & footers that appear *once* per page (which LZ4 and Gopher miss). |
| **JSD & Word Coverage** | Base-2 Jensen-Shannon Divergence (`0.0` = identical, `1.0` = disjoint) + token-weighted vocabulary coverage. | Measures whether a dataset's English register matches **GPT-2 (`OpenWebText`)** and our 4 target benchmarks. |
| **13-Gram Contamination** | Exact 13-word overlap between training documents and evaluation test splits. | Separates **healthy domain proximity** (teaching physics) from **cheating** (memorizing exam questions). |
| **WSD Annealing** | **Warmup → Stable → Decay**: 90% broad base mix, followed by 10% high-density educational/encyclopedic annealing. | Beats GPT-2 XL on *both* reasoning (**ARC-Easy**) and perplexity (**WikiText-103**, **LAMBADA**). |

---

### Big Picture: How This Audit Pipeline Works
Run **Step 1** below to sync the Kestrel repository and load our shared utilities, then inspect the end-to-end roadmap and live diagnostics step by step.


---
## Step 1 — Kaggle Setup, GitHub Repo Sync & Shared Utilities Import

### What Happens in This Cell (and Why)
When you import this notebook into **Kaggle**, Kaggle only copies the `.ipynb` file — not `src/kestrel/` or `docs/images/`.

This setup cell handles everything automatically:
1. Detects **Kaggle** (`/kaggle/working`) vs. local execution and installs any missing Python packages (`datasets`, `tiktoken`, `lz4`, `scipy`, `zstandard`, `seaborn`).
2. Checks local workspace paths for `src/kestrel`; if absent on Kaggle, shallow-clones **`https://github.com/MohammedKamalAlsyd/kestrel.git`** (default branch `master`) into `/kaggle/working/kestrel` and adds `src/` to `sys.path`.
3. Imports Kestrel's reusable utilities from **`kestrel.utils.notebook`** (`show_explainer_visual`, `configure_hf_cache`, `get_hf_token`) and diagnostics from **`kestrel.data.diagnostics`**.
   * `show_explainer_visual()` checks `docs/images/` **locally first** and falls back to **GitHub raw content** (`https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/...`) if absent on disk.

> **Kaggle Tip:** Turn **Internet ON** in the Kaggle sidebar (`Settings → Internet`) so Cell 1 can clone the repository and stream slices from Hugging Face Hub.


In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# 1. Auto-detect Kaggle vs. Local Workspace & Bootstrap Kestrel Repo
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/MohammedKamalAlsyd/kestrel.git"

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
SAMPLES_DIR = WORK_DIR / "samples"
FIGURES_DIR = SAMPLES_DIR / "figures"
CACHE_DIR = WORK_DIR / ".hf_cache"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Install Missing Dependencies Quietly
# ---------------------------------------------------------------------------
REQUIRED_PACKAGES = [
    ("datasets", "datasets>=3.0.0"),
    ("tiktoken", "tiktoken>=0.7.0"),
    ("lz4", "lz4>=4.3.0"),
    ("scipy", "scipy>=1.12"),
    ("zstandard", "zstandard>=0.22"),
    ("seaborn", "seaborn>=0.13"),
]
missing = []
for mod_name, pip_spec in REQUIRED_PACKAGES:
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pip_spec)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# ---------------------------------------------------------------------------
# 3. Locate or Auto-Clone Kestrel Repo (https://github.com/MohammedKamalAlsyd/kestrel)
# ---------------------------------------------------------------------------
REPO_ROOT = None
for candidate_root in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/input/kestrel"),
    WORK_DIR / "kestrel",
]:
    if (candidate_root / "src" / "kestrel" / "data" / "diagnostics.py").exists():
        REPO_ROOT = candidate_root.resolve()
        break

if REPO_ROOT is None:
    clone_dir = WORK_DIR / "kestrel"
    if (clone_dir / ".git").exists():
        print(f"Updating Kestrel repo at {clone_dir} ...")
        subprocess.check_call(["git", "-C", str(clone_dir), "pull", "--ff-only"])
    else:
        if clone_dir.exists() and any(clone_dir.iterdir()):
            shutil.rmtree(clone_dir, ignore_errors=True)
        print(f"Cloning Kestrel repo from {REPO_URL} -> {clone_dir} ...")
        res = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
            capture_output=True,
            text=True,
        )
        if res.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit {res.returncode}). "
                f"Ensure Kaggle Internet is ON and repo is public/accessible.\n"
                f"stderr: {res.stderr.strip()}"
            )
    REPO_ROOT = clone_dir.resolve()

src_path = str((REPO_ROOT / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ---------------------------------------------------------------------------
# 4. Import Shared Kestrel Utilities & Diagnostics
# ---------------------------------------------------------------------------
from kestrel.data import diagnostics as diag
from kestrel.utils.notebook import (
    configure_hf_cache,
    get_hf_token,
    show_explainer_visual,
)

configure_hf_cache(CACHE_DIR)
HF_TOKEN = get_hf_token("HF_TOKEN")
if HF_TOKEN:
    print("[OK] HF_TOKEN detected (gated repos such as Nemotron-CC v2.1 enabled).")
else:
    print("[INFO] No HF_TOKEN found; ungated datasets will run and gated Nemotron-CC v2.1 will be skipped cleanly.")

# ---------------------------------------------------------------------------
# 5. Unbiased Multi-Shard Sampling Budget (~15-25 MB JSONL per dataset)
# ---------------------------------------------------------------------------
TARGET_TOKENS = 3_500_000     # ~3.5M GPT-2 BPE tokens per dataset
MAX_DOCS = 5_000              # Cap at 5,000 docs per dataset
RANDOM_SAMPLE = True          # Multi-shard random sampling (avoids opening-shard prefix bias!)
NUM_SHARDS_TO_SAMPLE = 8      # Sample evenly across 8 randomly chosen remote shards
SHUFFLE_BUFFER_SIZE = 2_000   # Per-shard reservoir shuffle buffer
SEED = 42
FORCE_REFRESH = False         # Set True to re-sample and overwrite cached JSONL slices

print(f"[OK] Loaded kestrel from: {REPO_ROOT}")
print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'} | Work Dir: {WORK_DIR}")
print(
    f"Sampling    : {'Multi-shard random (' + str(NUM_SHARDS_TO_SAMPLE) + ' shards, buf=' + str(SHUFFLE_BUFFER_SIZE) + ')' if RANDOM_SAMPLE else 'Sequential prefix'} | "
    f"Budget: <= {MAX_DOCS:,} docs OR {TARGET_TOKENS:,} GPT-2 tokens"
)


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("01_kestrel_data_pipeline.png", "Figure 1: End-to-End Kestrel Data Selection Funnel — Multi-Shard Random Sampling, 3 Core Audits, and 2-Stage WSD Mixture.")


[Visual: Figure 1: End-to-End Kestrel Data Selection Funnel — Multi-Shard Random Sampling, 3 Core Audits, and 2-Stage WSD Mixture. — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/01_kestrel_data_pipeline.png]

---
## Step 2 — Multi-Shard Randomized Streaming & Full Audit Execution

### Why Multi-Shard Random Sampling Instead of Taking the First 5,000 Docs?
Taking only the opening rows of shard `00000` can introduce **prefix bias** (for example, if Common Crawl shards are grouped by crawl date, URL domain, or quality bucket).

To eliminate prefix bias while staying comfortably inside Kaggle's **20 GB disk / 30 GB RAM** limits:
1. **`stream_matched_slice(..., random_sample=True, num_shards_to_sample=8, shuffle_buffer_size=2_000)`** selects **8 distinct remote shards uniformly at random** across the entire multi-terabyte corpus, shuffles inside each shard with a 2,000-row buffer, and draws an equal token quota (`~target_tokens / 8`) from each shard.
2. **Need a whole-dataset sweep in constant RAM?** Use `diag.stream_aggregate_audit(spec, max_tokens=None, max_docs=None, random_sample=False)` on smaller corpora (or huge token budgets like 50M–100M tokens) to accumulate LZ4 histograms, Gopher repetition rates, lexical counts, and 13-gram contamination hits **on the fly without storing raw documents in memory**.

### The 6 Candidates We Audit
1. **`FineWeb-Edu (10BT)`** — Hugging Face's educational classifier-filtered Common Crawl baseline *(ODC-By v1.0)*.
2. **`DCLM-Baseline 1.0`** — FastText-filtered broad web corpus *(CC-BY-4.0)*. Strongest general linguistic diversity.
3. **`Nemotron-CC v2.1 (HQ)`** — NVIDIA's high-quality + synthetically rephrased web corpus *(Gated; requires `HF_TOKEN`)*.
4. **`Nemotron-ClimbMix (400B)`** *(2025 breakthrough)* — Semantic-cluster-bootstrapped mix tuned specifically with **62M–350M proxy models** *(Upstream CC-BY-NC-4.0)*.
5. **`FinePDFs-Edu (eng_Latn)`** *(2025–2026 breakthrough)* — High-density long-form educational PDF text *(ODC-By v1.0)*.
6. **`OpenWebText` (Control)** — GPT-2 original distribution anchor (**diagnostic control only**, never mixed into training).

Watch how low-RAM streaming and 13-gram sliding-window contamination scanning operate:


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("anim_01_streaming_and_13gram_scan.gif", "Animation 1: Low-RAM Equal-Token Streaming (Top) & 13-Gram Sliding-Window Contamination Scanner (Bottom).")


[Visual: Animation 1: Low-RAM Equal-Token Streaming (Top) & 13-Gram Sliding-Window Contamination Scanner (Bottom). — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/anim_01_streaming_and_13gram_scan.gif]

In [ ]:
import time
import pandas as pd

candidate_slices = {}
stream_summary_rows = []

print("=== 1/3 Multi-Shard Randomized Streaming of Candidate Datasets ===")
for key, spec in diag.CANDIDATE_DATASETS.items():
    cache_file = SAMPLES_DIR / f"{key}.jsonl"
    t0 = time.time()
    try:
        records = diag.stream_matched_slice(
            spec=spec,
            cache_path=cache_file,
            target_tokens=TARGET_TOKENS,
            max_docs=MAX_DOCS,
            hf_token=HF_TOKEN,
            force_refresh=FORCE_REFRESH,
            random_sample=RANDOM_SAMPLE,
            num_shards_to_sample=NUM_SHARDS_TO_SAMPLE,
            shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
            seed=SEED,
        )
        candidate_slices[key] = records
        elapsed = time.time() - t0
        n_tokens = sum(r["n_tokens"] for r in records)
        size_mb = cache_file.stat().st_size / (1024 * 1024)
        print(
            f"[OK]   {spec.display_name:<28s} | {len(records):>5,d} docs | "
            f"{n_tokens:>9,d} tokens | {size_mb:>5.1f} MB | {elapsed:>5.1f}s"
        )
        stream_summary_rows.append(
            {
                "Key": key,
                "Dataset": spec.display_name,
                "HF Repo": spec.repo_id,
                "License": spec.license,
                "Docs": len(records),
                "GPT-2 Tokens": n_tokens,
                "Mean Tok/Doc": round(n_tokens / max(len(records), 1), 1),
                "JSONL (MB)": round(size_mb, 2),
            }
        )
    except Exception as exc:
        print(f"[SKIP] {spec.display_name} ({spec.repo_id}): {exc}")

print("\n=== 2/3 Loading 100% of GPT-2 Target Evaluation Benchmarks ===")
eval_texts_by_benchmark = {}
for bkey, bspec in diag.EVAL_BENCHMARKS.items():
    eval_cache = SAMPLES_DIR / f"eval_{bkey}.jsonl"
    t0 = time.time()
    texts = diag.load_eval_benchmark_texts(
        spec=bspec,
        cache_path=eval_cache,
        force_refresh=FORCE_REFRESH,
    )
    eval_texts_by_benchmark[bkey] = texts
    print(f"[OK]   {bspec.display_name:<22s} | {len(texts):>6,d} examples/docs | {time.time() - t0:>4.1f}s")

display(pd.DataFrame(stream_summary_rows))

print("\n=== 3/3 Running Core Kestrel Diagnostic Audits ===")
t_audit = time.time()
audit_results = diag.run_full_audit(
    candidate_slices=candidate_slices,
    eval_texts_by_benchmark=eval_texts_by_benchmark,
    control_key="openwebtext",
)
print(f"[DONE] Full audit completed in {time.time() - t_audit:.1f}s across {len(candidate_slices)} datasets.")

# Release raw document strings immediately to keep Kaggle RAM minimal
del candidate_slices
gc.collect()


---
## Step 3A — Audit #1: LZ4 "Goldilocks Zone", Gopher Repetition & Cross-Doc Boilerplate

### Three Complementary, Non-Overlapping Quality Checks
1. **LZ4 Compression Ratio (`0.65 – 0.80` Goldilocks Zone):** Length-invariant information density check. `< 0.65` catches repetitive/templated text; `> 0.80` catches garbled OCR, hashes, or minified blobs.
2. **Gopher / FineWeb Within-Document Repetition ($n = 2\dots 10$):** Character-fraction normalized (`top_2..4gram`, `dup_5..10gram`), so long FinePDFs textbooks are never unfairly penalized.
3. **Cross-Document 7-Gram Boilerplate (`audit_cross_doc_boilerplate`):** Catches site-wide headers, cookie banners, and copyright footers that appear **once** per page (which single-document LZ4 & Gopher cannot see) — and prints the exact recurring strings so you can inspect or strip them before training.

### Visual Guide: What Each LZ4 Region Looks Like in Practice


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("03_lz4_goldilocks_explained.png", "Figure 2: The LZ4 'Goldilocks Zone' (0.65–0.80) Explained with Concrete Repetitive, Prose, and Garbled Examples.")


[Visual: Figure 2: The LZ4 'Goldilocks Zone' (0.65–0.80) Explained with Concrete Repetitive, Prose, and Garbled Examples. — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/03_lz4_goldilocks_explained.png]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.stats import gaussian_kde

sns.set_theme(style="whitegrid", context="notebook")

dataset_audits = audit_results["dataset_audits"]
alignment_matrix = audit_results["alignment_matrix"]
contamination_matrix = audit_results["contamination_matrix"]

ds_keys = list(dataset_audits.keys())
ds_labels = {
    k: diag.CANDIDATE_DATASETS[k].display_name if k in diag.CANDIDATE_DATASETS else k
    for k in ds_keys
}
palette = dict(zip(ds_keys, sns.color_palette("tab10", n_colors=max(len(ds_keys), 1))))

# 1. Overlaid LZ4 Compression Density + Gopher N-Gram Repetition Curves
fig, (ax_lz4, ax_rep) = plt.subplots(1, 2, figsize=(16, 5.5))

lo, hi = diag.GOLDILOCKS_LZ4_BAND
ax_lz4.axvspan(lo, hi, color="#059669", alpha=0.14, label=f"Goldilocks Band ({lo:.2f}–{hi:.2f})")
ax_lz4.axvline(lo, color="#059669", linestyle="--", linewidth=1.3)
ax_lz4.axvline(hi, color="#059669", linestyle="--", linewidth=1.3)

for dkey in ds_keys:
    ratios = np.array(dataset_audits[dkey]["compression_repetition"]["lz4_ratios"], dtype=np.float64)
    ratios_clipped = ratios[(ratios > 0.15) & (ratios < 1.15)]
    in_band_pct = 100.0 * dataset_audits[dkey]["compression_repetition"]["lz4_in_band_rate"]
    sns.kdeplot(
        ratios_clipped,
        ax=ax_lz4,
        color=palette[dkey],
        linewidth=2.2,
        label=f"{ds_labels[dkey]} ({in_band_pct:.1f}% in-band)",
    )

ax_lz4.set_title("Document-Level LZ4 Compression Ratio Density", fontsize=12, fontweight="bold")
ax_lz4.set_xlabel("LZ4 Ratio (Compressed / Raw Bytes) — Left: Repetitive | Right: Garbled")
ax_lz4.set_ylabel("Density")
ax_lz4.set_xlim(0.25, 1.05)
ax_lz4.legend(fontsize=8.5, loc="upper left")

ngram_cols = [f"top_{n}gram" for n in (2, 3, 4)] + [f"dup_{n}gram" for n in range(5, 11)]
x_labels = ["top-2", "top-3", "top-4", "dup-5", "dup-6", "dup-7", "dup-8", "dup-9", "dup-10"]
for dkey in ds_keys:
    means = dataset_audits[dkey]["compression_repetition"]["gopher_mean_rates"]
    flag_pct = 100.0 * dataset_audits[dkey]["compression_repetition"]["gopher_flag_rate"]
    vals = [100.0 * means.get(c, 0.0) for c in ngram_cols]
    ax_rep.plot(
        x_labels,
        vals,
        marker="o",
        linewidth=2.1,
        color=palette[dkey],
        label=f"{ds_labels[dkey]} (flag={flag_pct:.1f}%)",
    )

ax_rep.set_title("Gopher / FineWeb Within-Document N-Gram Repetition (%)", fontsize=12, fontweight="bold")
ax_rep.set_xlabel("Gopher N-Gram Metric (n = 2..10)")
ax_rep.set_ylabel("Mean Document Character Fraction (%)")
ax_rep.legend(fontsize=8.5, loc="upper right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_lz4_and_gopher_repetition.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Actionable Cross-Document Boilerplate Inspection (Top Recurring 7-Grams per Dataset)
print("=== Top Cross-Document 7-Gram Boilerplate Strings (Missed by Single-Doc Filters) ===")
for dkey in ds_keys:
    comp = dataset_audits[dkey]["compression_repetition"]
    rate = 100.0 * comp["cross_doc_boilerplate_doc_rate"]
    top_ngs = comp["top_cross_doc_ngrams"][:3]
    print(f"\n{ds_labels[dkey]} — {rate:.2f}% docs contain cross-doc boilerplate:")
    if top_ngs:
        for ng, df in top_ngs:
            print(f'   ({df:>4d} docs) "{ng}"')
    else:
        print("   [CLEAN] No 7-gram exceeded the cross-document boilerplate threshold.")

# 3. Live Interactive HTML5 Animation: Spotlight Each Audited Dataset's LZ4 Curve
x_grid = np.linspace(0.25, 1.05, 250)
kdes = {}
for dkey in ds_keys:
    arr = np.array(dataset_audits[dkey]["compression_repetition"]["lz4_ratios"], dtype=np.float64)
    arr = arr[(arr > 0.15) & (arr < 1.15)]
    if len(arr) > 5 and np.std(arr) > 1e-6:
        kdes[dkey] = gaussian_kde(arr)(x_grid)
    else:
        kdes[dkey] = np.zeros_like(x_grid)

max_y = max((float(v.max()) for v in kdes.values()), default=5.0) * 1.18

anim_fig, anim_ax = plt.subplots(figsize=(9.5, 4.2))
def _update_lz4_frame(frame_idx: int):
    anim_ax.clear()
    active_key = ds_keys[frame_idx]
    comp = dataset_audits[active_key]["compression_repetition"]

    anim_ax.axvspan(lo, hi, color="#059669", alpha=0.14)
    anim_ax.axvline(lo, color="#059669", linestyle="--", lw=1.2)
    anim_ax.axvline(hi, color="#059669", linestyle="--", lw=1.2)

    for dk in ds_keys:
        if dk != active_key:
            anim_ax.plot(x_grid, kdes[dk], color="#94a3b8", lw=1.2, alpha=0.4)

    anim_ax.plot(x_grid, kdes[active_key], color=palette[active_key], lw=3.0, label=ds_labels[active_key])
    anim_ax.fill_between(x_grid, kdes[active_key], color=palette[active_key], alpha=0.22)

    anim_ax.set_xlim(0.25, 1.05)
    anim_ax.set_ylim(0, max_y)
    anim_ax.set_xlabel("LZ4 Compression Ratio (Goldilocks Band = 0.65–0.80)")
    anim_ax.set_ylabel("Density")
    anim_ax.set_title(
        f"Interactive Spotlight [{frame_idx + 1}/{len(ds_keys)}]: {ds_labels[active_key]}  |  "
        f"In-Band: {100 * comp['lz4_in_band_rate']:.1f}%  |  "
        f"Below (<0.65): {100 * comp['lz4_below_band_rate']:.1f}%  |  "
        f"Above (>0.80): {100 * comp['lz4_above_band_rate']:.1f}%",
        fontsize=10.5,
        fontweight="bold",
    )
    anim_ax.legend(loc="upper left")

anim = FuncAnimation(anim_fig, _update_lz4_frame, frames=len(ds_keys), interval=1400, repeat=True)
plt.close(anim_fig)
print("\nInteractive Dataset-by-Dataset LZ4 Inspector (click Play or Step below):")
display(HTML(anim.to_jshtml()))


---
## Step 3B — Audits #2 & #3: Healthy Domain Proximity (JSD & Coverage) vs. Cheating (13-Gram Leakage)

### Separating Genuine Learning from Memorization
* **Healthy Domain Proximity (What We WANT):** Your training corpus teaches the *same vocabulary, grammar, and scientific principles* tested by **WikiText-103**, **LAMBADA**, **HellaSwag**, and **ARC-Easy** → measured by **low Jensen-Shannon Divergence (JSD)** and **high token-weighted common-word coverage**.
* **Benchmark Contamination (What We FORBID):** Your training corpus literally contains *verbatim test questions* scraped from Quizlet, GitHub, or Wikipedia mirrors → caught by our **13-gram exact-overlap detector** (`audit_contamination_13gram`).

### Visual Guide: How to Tell Genuine Learning Apart from Memorization


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("04_proximity_vs_contamination.png", "Figure 3: Healthy Domain Proximity (Low JSD, 0% Leakage) vs. Benchmark Cheating (Exact 13-Gram Overlap).")


[Visual: Figure 3: Healthy Domain Proximity (Low JSD, 0% Leakage) vs. Benchmark Cheating (Exact 13-Gram Overlap). — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/04_proximity_vs_contamination.png]

In [ ]:
ref_keys = (["openwebtext"] if "openwebtext" in alignment_matrix[ds_keys[0]] else []) + list(
    eval_texts_by_benchmark.keys()
)
ref_labels = {
    "openwebtext": "OpenWebText\n(GPT-2 Ctrl)",
    **{k: v.display_name.replace(" ", "\n") for k, v in diag.EVAL_BENCHMARKS.items()},
}

jsd_df = pd.DataFrame(
    {
        ref_labels[rk]: [alignment_matrix[dk][rk]["unigram_jsd"] for dk in ds_keys]
        for rk in ref_keys
    },
    index=[ds_labels[dk] for dk in ds_keys],
)

cov_df = pd.DataFrame(
    {
        ref_labels[rk]: [100.0 * alignment_matrix[dk][rk]["common_word_coverage"] for dk in ds_keys]
        for rk in ref_keys
    },
    index=[ds_labels[dk] for dk in ds_keys],
)

eval_only_keys = list(eval_texts_by_benchmark.keys())
contam_df = pd.DataFrame(
    {
        ref_labels[bk]: [
            100.0 * contamination_matrix[dk][bk]["eval_contamination_rate"] for dk in ds_keys
        ]
        for bk in eval_only_keys
    },
    index=[ds_labels[dk] for dk in ds_keys],
)

fig, axes = plt.subplots(1, 3, figsize=(20, 0.85 * len(ds_keys) + 3.2))

sns.heatmap(
    jsd_df,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    ax=axes[0],
    cbar_kws={"label": "Unigram JSD (bits, lower = closer)"},
)
axes[0].set_title("Unigram Jensen-Shannon Divergence\n(Lower = Closer Domain Distribution)", fontweight="bold")

sns.heatmap(
    cov_df,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=axes[1],
    cbar_kws={"label": "Common-Word Coverage (%)"},
)
axes[1].set_title("Token-Weighted Common-Word Coverage (%)\n(Higher = Stronger Shared Vocabulary)", fontweight="bold")
axes[1].set_yticklabels([])

sns.heatmap(
    contam_df,
    annot=True,
    fmt=".3f",
    cmap="Reds",
    vmin=0.0,
    ax=axes[2],
    cbar_kws={"label": "Eval Examples Hit (%)"},
)
axes[2].set_title("13-Gram Exact-Overlap Contamination (%)\n(Target: ~0.0% Verbatim Leakage)", fontweight="bold")
axes[2].set_yticklabels([])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_alignment_and_contamination_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Step 4 — Summary Scorecard & How We Turn Audits into Kestrel's 2-Stage Mixture

### How the Composite Score (0–100) Works (Zero Fluff, Strictly High-Signal)
Every dataset is scored transparently in `diag.build_scorecard()` using three non-overlapping pillars:
* **45% Cleanliness (`1 - Noise Rate`):** Penalizes LZ4 out-of-band documents (`50%`), Gopher n-gram repetition flags (`35%`), and cross-document 7-gram boilerplate (`15%`).
* **40% GPT-2 Benchmark Alignment:** Combines low Unigram/Bigram JSD and high token-weighted common-word coverage across `HellaSwag`, `ARC-Easy`, `LAMBADA`, and `WikiText-103`.
* **15% Decontamination Safety:** Penalizes datasets whose max 13-gram eval overlap approaches 5%.

### Crucial Senior Researcher Insight: Why We Mix Instead of Picking Only #1
Even if one corpus ranks #1 overall, training **100%** on a single dataset triggers a classic small-LM trade-off:
* **100% Educational (`FineWeb-Edu` / `FinePDFs-Edu`)** crushes **ARC-Easy** science reasoning, but drifts away from casual narrative/commonsense prose (**LAMBADA**, **HellaSwag**).
* **100% Web (`DCLM-Baseline`)** nails **LAMBADA** and **HellaSwag** fluency, but lacks the textbook density a 350M–700M model needs to beat 1.5B GPT-2 XL on **ARC-Easy** and **WikiText-103**.

That's why Kestrel uses a **Two-Stage WSD (Warmup–Stable–Decay) Curriculum**:
* **Stage 1 (0–90% of tokens):** Broad, high-density foundation (`32% DCLM` + `25% FinePDFs-Edu` + `23% FineWeb-Edu` + `15% Nemotron-CC HQ` + `5% Code/Math`).
* **Stage 2 Annealing (Final 10% during LR decay):** Sharpens factual reasoning & encyclopedic fluency (`25% FineWeb-Edu >=4` + `25% Dolmino` + `20% FinePDFs-Edu` + `15% Cosmopedia-v2`) while keeping a **15% `DCLM-Baseline` anchor** so narrative perplexity never regresses!

Watch how Kestrel's 2-stage mixture evolves over training in the animation below:


In [ ]:
# Uses shared kestrel.utils.notebook helper (local docs/images/ first -> GitHub raw fallback)
show_explainer_visual("anim_02_wsd_curriculum_mixture.gif", "Animation 2: Kestrel's Two-Stage WSD Curriculum (90% Base Mix → 10% High-Density Annealing) vs. GPT-2 XL.")


[Visual: Animation 2: Kestrel's Two-Stage WSD Curriculum (90% Base Mix → 10% High-Density Annealing) vs. GPT-2 XL. — https://raw.githubusercontent.com/MohammedKamalAlsyd/kestrel/master/docs/images/anim_02_wsd_curriculum_mixture.gif]

In [ ]:
scorecard = audit_results["scorecard"].copy()

display_cols = [
    "Rank",
    "Dataset",
    "Docs",
    "Tokens",
    "LZ4 In-Band (%)",
    "Gopher Rep Flag (%)",
    "Cross-Doc Boilerplate (%)",
    "Noise Rate (%)",
    "OWT Unigram JSD",
    "Eval Mean JSD (Uni)",
    "Eval Word Coverage (%)",
    "GPT-2 Benchmark Alignment",
    "13-gram Contam Mean (%)",
    "13-gram Contam Max (%)",
    "Composite Score",
]
display_cols = [c for c in display_cols if c in scorecard.columns]

print("=== Kestrel Pretraining Dataset Audit Scorecard ===")
styled = (
    scorecard[display_cols]
    .style.background_gradient(subset=["Composite Score", "GPT-2 Benchmark Alignment"], cmap="Greens")
    .background_gradient(subset=["Noise Rate (%)", "13-gram Contam Max (%)"], cmap="Reds")
    .format(precision=3, na_rep="—")
)
display(styled)

# Transparently inspect any exact 13-gram overlaps
print("\n=== Sample 13-Gram Exact Matches (Human Contamination Inspection) ===")
any_hits = False
for dk in ds_keys:
    for bk, cres in contamination_matrix[dk].items():
        if cres["matched_unique_ngrams"] > 0:
            any_hits = True
            print(
                f"  - {ds_labels[dk]} x {bk}: {cres['contaminated_eval_examples']}/{cres['eligible_eval_examples']} "
                f"eval examples ({100 * cres['eval_contamination_rate']:.3f}%), "
                f"{cres['contaminated_candidate_docs']} candidate docs"
            )
            for snippet in cres["sample_matches"][:3]:
                print(f'      match: "{snippet}"')
if not any_hits:
    print("  [CLEAN] Zero exact 13-gram overlaps detected across the audited slices.")

# Export scorecard and compact JSON summary for Kestrel training logs
scorecard_csv = SAMPLES_DIR / "scorecard.csv"
scorecard.to_csv(scorecard_csv, index=False)

compact_summary = {
    "slice_config": {
        "target_tokens": TARGET_TOKENS,
        "max_docs": MAX_DOCS,
        "random_sample": RANDOM_SAMPLE,
        "num_shards_to_sample": NUM_SHARDS_TO_SAMPLE,
        "shuffle_buffer_size": SHUFFLE_BUFFER_SIZE,
        "seed": SEED,
    },
    "scorecard": scorecard.to_dict(orient="records"),
    "alignment_matrix": alignment_matrix,
    "contamination_summary": {
        dk: {
            bk: {k: v for k, v in cres.items() if k != "sample_matches"}
            for bk, cres in bmap.items()
        }
        for dk, bmap in contamination_matrix.items()
    },
}
summary_json = SAMPLES_DIR / "audit_summary.json"
summary_json.write_text(json.dumps(compact_summary, indent=2), encoding="utf-8")

print(f"\nSaved scorecard CSV : {scorecard_csv}")
print(f"Saved summary JSON  : {summary_json}")
print(f"Saved figures under : {FIGURES_DIR}")
print("\nFull mixture justification & licensing tracks: docs/dataset_justification.md")
